# NB-06 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-06**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-06.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This reproduces the retained testbed variance comparisons and explains why variance compression can omit locally important congestion risks.

- [Volume IV, Chapter 18 — PCA: compression versus physical identity](https://sherandoresearch.github.io/power-markets-series/vol4_testbed_atlas.html#chiv-18)
- [Volume IV, Chapter 21 — FTRs as a network-derivative laboratory](https://sherandoresearch.github.io/power-markets-series/vol4_testbed_atlas.html#chiv-21)
- [Volume III, Chapter 2 — Topological modes, statistical factors, and physics atoms](https://sherandoresearch.github.io/power-markets-series/vol3_spectral_grids.html#c2)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-06', require_key=True)
print(setup_status)


# NB-06 — Modes, PCA, covariance, and rare local risk
Companion to Volume IV §§11–12,15–20 and §21.2–21.4. Requires basic covariance
and projection algebra. Reuse the retained market year and corrected FTR
artifacts through authenticated private method calls. No simulation, random draws,
surrogates, or new observations are generated.

We will reproduce the variance ledger, measure omitted-band errors, and
distinguish graph modes, statistical factors, and physical constraint atoms.

In [ ]:
NOTEBOOK_ID='NB-06'
from methods_notebooks.session import digest

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return {k:response[k] for k in ('result','diagnostics')}
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

print('Local original price inspection:',inputs['requests']['modal_risk']['request']['inputs']['graph_signal']['values'][:1])
print('Private source-span preparation:',inputs['comparison_basis_preparation'])
def call(name,parents=(),request=None):
    entry=inputs['requests'][name]
    return execute(name,entry['method'],entry['request'] if request is None else request,parents)
modal=call('modal_risk')['result']['reconstruction_diagnostics']
labels={p['path_id']:{'path_id':p['path_id'],'source':p['source_label'],'sink':p['sink_label'],'role':p['role']} for p in inputs['path_labels']}
ledger=[dict(labels[r['path_id']],**{k:v for k,v in r.items() if k!='path_id'}) for r in modal['ledger']]
for row in ledger[:3]:
    print({k:row[k] for k in ('path_id','diagonal_modal_variance_allocation','low_projection_variance_ratio',
        'omitted_projection_variance_ratio','twice_cross_covariance_ratio','truncation_bias','truncation_mse','tail_rmse')})

## The covariance term cannot be discarded
On intact hours, each path satisfies Var(Y)=Var(Ylow)+Var(Yhigh)+2Cov.
The diagonal allocation is a different statistic. For N1→S5 it exceeds 99%,
yet the omitted projection contributes roughly 28% of full variance before
cross-covariance cancellation. N8→S3's low-projection variance even exceeds
total variance. Neither fact is a numerical error.

In [ ]:
parts=np.array([[r[k] for k in ('low_projection_variance_ratio','omitted_projection_variance_ratio','twice_cross_covariance_ratio')] for r in ledger[:3]])
np.testing.assert_allclose(parts.sum(axis=1),1,atol=1e-9)
assert ledger[0]['diagonal_modal_variance_allocation']>.99
assert ledger[0]['omitted_projection_variance_ratio']>.25
assert ledger[1]['low_projection_variance_ratio']>1
fig,ax=plt.subplots(figsize=(10,4))
x=np.arange(3)
for k,(label,color) in enumerate(zip(['Low variance / full','Omitted variance / full','2 × covariance / full'],['#88C0D0','#EBCB8B','#BF616A'])):
    ax.bar(x+(k-1)*.24,parts[:,k],width=.24,label=label,color=color)
ax.axhline(0,color='#D8DEE9',linewidth=.6)
ax.set_xticks(x,['N1→S5','N8→S3','S5→S9']);ax.set(ylabel='Ratio to full variance',title='Variance accounting includes signed covariance')
ax.legend(facecolor='#3B4252',labelcolor='#ECEFF4',loc='center left',bbox_to_anchor=(1,0.5),fontsize=9);fig.tight_layout();plt.show()

## Rare local risk survives a compact representation
The local S5→S9 path has little low-mode exposure. Inspect bias, squared error,
and RMSE during its largest absolute realized payoffs (top 5%, ties retained).
The threshold is retrospective diagnostic information, not a trading signal.
Counts below describe the existing controlled year's support, not independent
tail observations or a confidence interval.

In [ ]:
local=2;series=modal['selected_series'][0]
assert series['path_id']==ledger[local]['path_id']
active=np.asarray(inputs['local_constraint_active'],bool)
print({'intact_hours':len(modal['sample_indices']),'local_constraint_dual_active_hours':int(active.sum()),
       'local_path_tail_hours':ledger[local]['tail_hours'],'local_path_abs_error_p95':ledger[local]['abs_error_p95'],
       'local_path_tail_rmse':ledger[local]['tail_rmse']})
fig,ax=plt.subplots(figsize=(8,4))
ax.scatter(np.asarray(series['payoff']),np.asarray(series['low_projection']),s=9,alpha=.4,color='#88C0D0')
limits=[float(np.asarray(series['payoff']).min()),float(np.asarray(series['payoff']).max())]
ax.plot(limits,limits,'--',color='#EBCB8B',label='Exact reconstruction')
ax.set(xlabel='Actual S5→S9 payoff (USD/MWh)',ylabel='Two-mode projection (USD/MWh)',title='Low modes miss the local path tail')
ax.legend(facecolor='#3B4252',labelcolor='#ECEFF4');fig.tight_layout();plt.show()

## PCA must share coordinates with the comparison span
Aggregate to the existing 365 daily sums, fit means, standard deviations,
and PCA on the first 250 days, and inspect the remaining 115 days separately.
Physical atoms and graph path coordinates are divided by the same training
standard deviations. Rank-revealing bases remove redundant atoms. A small
principal angle alone does not mean two spaces have the same dimension.

In [ ]:
pca_output=call('training_pca');fitted=pca_output['result']['pca_model_summary']
for key in ('mean','scale','active_mask'):
    # Prepared physical/graph spans are valid only in the exact source scaling.
    np.testing.assert_array_equal(fitted[key],inputs['expected_training_scaling'][key])
pca={'mean':np.asarray(fitted['mean']),'sd':np.asarray(fitted['scale']),
    'keep':np.asarray(fitted['active_mask']),'values':np.asarray(fitted['eigenvalues'])}
spans={}
for name in ('physical_two_topology','two_graph_modes_intact'):
    req=copy.deepcopy(inputs['requests'][name]['request'])
    left=req['inputs']['basis_pair']['left']
    left['values']=fitted['retained_components']
    left['operator_sha256']=digest({'values':left['values'],'carrier_context':left['carrier_context']})
    result=call(name,['training_pca'],req)['result']['basis_alignment']
    assert not result['coordinate_transport_performed'] and result['procrustes_rotation'] is None
    spans[name]={'rank':result['right_dimension'],'angles_degrees':np.degrees(result['principal_angles_radians']).tolist(),
        'pca_to_span_relative_residual':result['left_to_right_relative_residual']}
    print(name,spans[name])
print({'train_days':250,'heldout_days':115,
       'first_two_training_variance_share':fitted['explained_variance_fraction'],
       'heldout_standardized_mse_two_pcs':pca_output['diagnostics']['evaluation_standardized_mse']})
# The original reversed-evaluation fit check was independently replayed in the
# private source kernel proof. It is not represented as a separate REST call.
print('Original fit-order control:',inputs['original_fit_order_control'])

## White-noise reference versus the existing heterogeneous forcing
Reuse the heterogeneous noise levels and bus locations specified in
`analytics.py`, solving the OU covariance analytically on the nonconstant
graph subspace. The homogeneous reference uses its existing background noise
level at every bus. These are covariance calculations, not generated samples.
White forcing is diagonal in the graph basis; heterogeneous forcing generally
creates cross-modal covariance. This state model is not a price model.
The heatmap uses the current returned graph basis and its recorded sign convention.
Mode signs may differ from the original North-mean convention; the underlying
nodal covariance and invariant subspaces are checked independently.

In [ ]:
noise={}
for label in ('homogeneous_reference','existing_heterogeneous'):
    field=call('noise:'+label)['result']['field_covariance']
    assert field['stationary_scope']=='mean_zero_subspace_only' and field['null_mode_stationary_variance']=='not_claimed'
    noise[label]={'modal_covariance':np.asarray(field['modal_covariance'])[1:,1:],
        'off_diagonal_fraction':field['off_diagonal_modal_fraction']}
print(inputs['plot_basis_policy'])
for label,item in noise.items(): print(label,{'off_diagonal_covariance_fraction':item['off_diagonal_fraction']})
assert noise['homogeneous_reference']['off_diagonal_fraction']<1e-10
assert noise['existing_heterogeneous']['off_diagonal_fraction']>1e-4
# Normalize each covariance to correlation to expose off-diagonal structure.
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,(name,item) in zip(axes,noise.items()):
    cov=item['modal_covariance'];sd=np.sqrt(np.diag(cov));corr=cov/sd[:,None]/sd[None,:]
    im=ax.imshow(corr,vmin=-1,vmax=1,cmap='RdBu');ax.set(title=name.replace('_',' '),xlabel='Nonconstant graph mode',ylabel='Nonconstant graph mode')
    ticks=[0,4,9,14,19,24]
    ax.set_xticks(ticks,[1,5,10,15,20,25]);ax.set_yticks(ticks,[1,5,10,15,20,25])
fig.colorbar(im,ax=axes.ravel().tolist(),shrink=.8,label='Modal correlation');plt.show()

## An MP edge is a reference, not a count of physical constraints
Recompute descriptive correlation spectra using nonoverlapping 1-, 5-, and
10-day blocks of the same training period. Each aggregation changes the target
horizon as well as sample count. These rows demonstrate sensitivity; they do
not calibrate a dependence-aware null or prove independent observations.
No retained surrogate/null ensemble is available for this notebook. Producing
a new synthetic null ensemble would require separate approval.

In [ ]:
sensitivity=[]
for name,days in [('training_daily',1),('training_five_day',5),('training_ten_day',10)]:
    diagnostic=call(name)['result']['reconstruction_diagnostics']
    row={key:diagnostic[key] for key in ('observations','products','iid_mp_reference','eigenvalues_above_reference')}
    row['block_days']=days;sensitivity.append(row);print(row)
print('Daily training PC1 lag-one correlation:',pca_output['diagnostics']['training_first_component_lag_one_correlation'])
published_result=call('published_full_sample')['result']
published={'eigenvalues':published_result['pca_model_summary']['eigenvalues'],
    'spike_count':published_result['reconstruction_diagnostics']['eigenvalues_above_reference'],
    'structural_comparison_rank':spans['physical_two_topology']['rank'],
    'threshold_interpretation':inputs['published_reference']['threshold_interpretation']}
assert published['spike_count']==inputs['published_reference']['spike_count']
assert published['structural_comparison_rank']==inputs['published_reference']['structural_comparison_rank']
print('Published full-sample reference:',{k:published[k] for k in ('spike_count','structural_comparison_rank','threshold_interpretation')})
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(np.arange(1,len(pca['values'])+1),pca['values'],'o-',color='#88C0D0',label='Training correlation spectrum')
ax.axhline(sensitivity[0]['iid_mp_reference'],color='#EBCB8B',linestyle='--',label='iid MP reference only')
ax.set_xticks([1,3,6,9,12,15,18])
ax.set(xlabel='PCA component',ylabel='Correlation eigenvalue',title='Training-only spectrum; threshold is not an inference test')
ax.legend(facecolor='#3B4252',labelcolor='#ECEFF4');fig.tight_layout();plt.show()

In [ ]:
for name in ('training_pca','physical_two_topology','modal_risk','published_full_sample'):
    entry=inputs['requests'][name];bad=copy.deepcopy(entry['request'])
    if name=='training_pca':bad['inputs']['train_eval_panel']['evaluation']['sample_indices'][0]=249
    elif name=='physical_two_topology':bad['inputs']['basis_pair']['right']['carrier_context']='incompatible'
    elif name=='modal_risk':bad['inputs']['graph_signal']['excluded_sample_indices'].pop()
    else:bad['inputs']['aligned_panel']['end_index_exclusive']-=1
    reject(entry['method'],bad)
checked={'training_pca':{'mean':pca['mean'].tolist(),'sd':pca['sd'].tolist(),'keep':pca['keep'].tolist(),'eigenvalues':pca['values'].tolist()},
    'variance_ledger':ledger,'training_pca_spans':spans,'dependence_sensitivity':sensitivity,
    'analytic_noise_off_diagonal_fraction':{k:v['off_diagonal_fraction'] for k,v in noise.items()},
    'training_days':250,'evaluation_days':115,'intact_hours':len(modal['sample_indices']),
    'plot_basis_policy':inputs['plot_basis_policy'],'scientific_acceptance':False}
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
print(session.finish(checked))

## Exercise and what failed
Explain why the N8→S3 low-band variance can exceed full variance, and why a
near-complete diagonal allocation did not protect N1→S5 from residual error.
Inspect the negative cross-covariance term in the ledger for the answer.
A compact representation preserves some variance, not every path's tail.
Neither PCA eigenvalue counts nor an iid MP reference identify physical
constraints or establish forecasting skill.